# La Plata: cargos ejecutivos, 2011-2023 (Presidente, Gobernador, Intendente)

La Plata, Provincia de Buenos Aires, Generales, para los tres cargos ejecutivos — **Presidente**, **Gobernador** e **Intendente** — a lo largo de cuatro elecciones: **2011, 2015, 2019 y 2023**. Cubre todo el ciclo: traer el CSV oficial de cada (año, categoría), ver el resultado, validarlo contra el agregado JSON, un ejemplo de análisis mesa por mesa, y confirmar el estado final del caché en disco.

Usamos `resultado/totalizadocsv` — el endpoint que arma el CSV oficial descargable del sitio (`GET /api/resultado/totalizadocsv`, con parámetros `año`, `recuento`, `idEleccion`, `idCargo`, `idDistrito`, `idSeccionProvincial`, `idSeccion`). Trae todas las mesas de la categoría en un solo pedido.

El `categoriaId`/`idCargo` de cada categoría se resolvió probando valores y leyendo el campo `cargo_nombre` que devuelve el propio CSV (no hay forma de derivarlo de otra manera). Para La Plata, el mapeo resultó **estable en los cuatro años**:

| categoriaId | cargo |
|---|---|
| 1 | PRESIDENTE |
| 4 | GOBERNADOR |
| 7 | INTENDENTE |


In [ ]:
import io
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient
from electoral.models import ResultadoElectoral

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data")

ANIOS = [2011, 2015, 2019, 2023]

CATEGORIAS = {
    "presidente": 1,
    "gobernador": 4,
    "intendente": 7,
}

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Traer el CSV oficial de cada (año, categoría)

In [ ]:
dataframes = {}
for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        csv_bytes = client.get_resultados_csv(
            anio_eleccion=anio, categoria_nombre=nombre, categoria_id=categoria_id, **LA_PLATA
        )
        df = pd.read_csv(io.BytesIO(csv_bytes))
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        dataframes[(anio, nombre)] = df
        print(f"{anio}/{nombre}: {len(df)} filas, {df['mesa_id'].nunique()} mesas, "
              f"mesa_tipo={sorted(df['mesa_tipo'].unique())}, cargo_nombre={df['cargo_nombre'].unique()}")

## 2. Resultado de cada (año, categoría)

In [ ]:
for (anio, nombre), df in dataframes.items():
    positivos = (
        df[df["votos_tipo"] == "POSITIVO"]
        .groupby("agrupacion_nombre")["votos_cantidad"]
        .sum()
        .sort_values(ascending=False)
    )
    print(f"=== {anio}/{nombre} ===")
    print(positivos.head(5).to_string())
    print()

## 3. Validar contra el agregado de la API (JSON)

Sumamos los positivos del CSV por agrupación y los comparamos contra `client.get_resultados` (el agregado JSON, un solo pedido adicional por categoría). A diferencia de los notebooks anteriores, acá **no asumimos que siempre van a coincidir** — ya sabemos que el agregado JSON puede subestimar (visto antes en CABA/2019, y de nuevo en La Plata/2019/Presidente). El CSV es la fuente confiable; esta validación sirve para detectar en qué (año, categoría) el agregado JSON no se puede usar.

In [ ]:
def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        df = dataframes[(anio, nombre)]
        positivos_csv = (
            df[df["votos_tipo"] == "POSITIVO"]
            .groupby("agrupacion_id")["votos_cantidad"]
            .sum()
        )
        positivos_csv.index = positivos_csv.index.map(lambda x: normalizar_id(x))

        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=nombre, categoria_id=categoria_id, **LA_PLATA
        )
        resultado = ResultadoElectoral.from_json(raw)
        positivos_api = {
            normalizar_id(a.id_agrupacion): a.votos
            for a in resultado.valores_totalizados_positivos
        }

        agrupaciones = sorted(set(positivos_csv.index) | set(positivos_api))
        diffs = [ag for ag in agrupaciones if positivos_csv.get(ag, 0) != positivos_api.get(ag, 0)]
        estado = "OK" if not diffs else f"AGREGADO JSON NO CONFIABLE (difiere en {len(diffs)} agrupaciones)"
        print(f"{anio}/{nombre}: {estado}")

## 4. Análisis mesa por mesa, directo del CSV (ejemplo: La Plata/Presidente/2011)

El CSV ya trae la columna `mesa_id`: alcanza para cualquier análisis mesa por mesa sin pedirle nada más a la API. Ejemplo: participación por mesa y votos del ganador por mesa.

In [ ]:
df_2011_presidente = dataframes[(2011, "presidente")]

participacion = df_2011_presidente.groupby("mesa_id").agg(
    votantes=("votos_cantidad", "sum"),
    electores=("mesa_electores", "first"),
    circuito=("circuito_id", "first"),
)
participacion["participacion_pct"] = 100 * participacion["votantes"] / participacion["electores"]

print("mesas con mayor participación:")
print(participacion.sort_values("participacion_pct", ascending=False).head(5))
print("\nmesas con menor participación:")
print(participacion.sort_values("participacion_pct").head(5))

ganador = (
    df_2011_presidente[df_2011_presidente["votos_tipo"] == "POSITIVO"]
    .groupby("agrupacion_nombre")["votos_cantidad"]
    .sum()
    .idxmax()
)
votos_ganador_por_mesa = (
    df_2011_presidente[
        (df_2011_presidente["votos_tipo"] == "POSITIVO")
        & (df_2011_presidente["agrupacion_nombre"] == ganador)
    ]
    .set_index("mesa_id")["votos_cantidad"]
    .sort_index()
)
print(f"\nganador 2011/presidente: {ganador}")
print("votos del ganador, primeras 5 mesas:")
print(votos_ganador_por_mesa.head(5))

## 5. Estado final del caché en disco

Cada (año, categoría) debería tener exactamente 2 archivos: el agregado (`.json`) y el CSV oficial (`.csv`). Nada de JSON por mesa acumulado.

In [ ]:
ok = True
for anio in ANIOS:
    for nombre in CATEGORIAS:
        archivos = sorted(p.name for p in (REPO / "data" / str(anio) / nombre).iterdir())
        if len(archivos) != 2:
            ok = False
            print(f"{anio}/{nombre}: ¡{len(archivos)} archivos! {archivos}")

print("OK: todas las carpetas tienen exactamente 2 archivos." if ok else "hay carpetas con archivos de más")

## 6. Tabla de agrupaciones por año y nivel

`data/agrupaciones/agrupaciones.csv`: una fila por (año, agrupación, nivel) — qué agrupaciones compitieron en cada nivel de cargo en cada elección. Sale del agregado JSON (no del CSV): trae la misma lista de agrupaciones con muchísimo menos texto que parsear (el agregado tiene una fila por agrupación; el CSV tiene una fila por mesa×agrupación×tipo de voto). Verificado que la lista de agrupaciones no se ve afectada por el bug de conteo del agregado JSON (2019/presidente): las agrupaciones que aparecen son las mismas en JSON y en CSV, aunque los votos totales del agregado estén mal — el bug subestima mesas, no hace desaparecer agrupaciones del padrón de listas.

`nivel` usa el nombre que pidieron: `presidente` / `gobernacion` / `intendente` (no `gobernador`).

In [ ]:
NIVELES = {"presidente": "presidente", "gobernador": "gobernacion", "intendente": "intendente"}

filas_agrupaciones = []
for anio in ANIOS:
    for nombre, categoria_id in CATEGORIAS.items():
        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=nombre, categoria_id=categoria_id, **LA_PLATA
        )
        resultado = ResultadoElectoral.from_json(raw)
        for a in resultado.valores_totalizados_positivos:
            filas_agrupaciones.append(
                {"anio": anio, "agrupacion": a.nombre_agrupacion, "nivel": NIVELES[nombre]}
            )

df_agrupaciones = (
    pd.DataFrame(filas_agrupaciones)
    .drop_duplicates()
    .sort_values(["anio", "nivel", "agrupacion"])
    .reset_index(drop=True)
)

destino = REPO / "data" / "agrupaciones"
destino.mkdir(parents=True, exist_ok=True)
df_agrupaciones.to_csv(destino / "agrupaciones.csv", index=False)

print(f"{len(df_agrupaciones)} filas -> {destino / 'agrupaciones.csv'}")
df_agrupaciones.head(10)